In [254]:
from pathlib import Path
import json
import pandas as pd

In [255]:
RETRIEVAL_DIR = Path("../output/retrieval")
DATA_DIR = Path("../data/gold_data")

METHOD = "hybrid_rerank" # "bm25" # # "bm25_rerank" # "dense" # "dense_instruct" # "dense_instruct_rerank" # "dense_rerank" # hybrid # "hybrid_instruct" # "hybrid_instruct_rerank" # "hybrid_rerank"

RESULTS_DIR = Path(f"../results/retrieval/{METHOD}")
RESULTS_DIR.mkdir(parents=False, exist_ok=True)

In [256]:
# Evaluate either with or w/o graphics
NO_GRAPHICS = True

if NO_GRAPHICS:
    gold_processed_df = pd.read_csv(DATA_DIR / "gold_standard_processed_no_graphics.csv")
else:
    gold_processed_df = pd.read_csv(DATA_DIR / "gold_standard_processed.csv")

In [257]:
print(f"Gold before filtering NA: {len(gold_processed_df)}")
gold_processed_df = gold_processed_df.dropna(subset=["page"])
print(f"Gold after filtering NA: {len(gold_processed_df)}")

Gold before filtering NA: 617
Gold after filtering NA: 547


In [258]:
gold_pages_by_report = gold_processed_df.groupby("report_id")["page"].agg(set).to_dict()
gold_pages_by_report

{'Allianz_2022': {79.0},
 'Daimler_2020': {63.0},
 'Fresenius SE_2019': {126.0},
 'acuity brands inc_2022': {99.0, 101.0},
 'addtech_2022': {73.0},
 'aixtron_2020': {51.0},
 'allfunds group_2021': {125.0, 142.0},
 'americold realty inc_2022': {53.0},
 'apollo commercial real estate fina_2019': {9.0},
 'arteria networks corp_2019': {10.0},
 'autoneum holding_2019': {27.0},
 'balanced commercial property trust_2020': {34.0},
 'bekaert (d) sa_2022': {276.0},
 'blackberry ltd_2022': {21.0, 22.0},
 'blackline inc_2021': {21.0},
 'cabot corp_2018': {22.0},
 'cardinal energy ltd_2021': {19.0},
 'century communities inc_2019': {11.0},
 'chase corp_2020': {32.0},
 'comerica inc_2019': {44.0},
 'cubesmart reit_2021': {19.0},
 'cytokinetics inc_2022': {24.0},
 'dampskibsselskabet norden_2019': {63.0},
 'dksh holding_2021': {73.0, 74.0},
 'evolution mining ltd_2020': {44.0},
 'fabege_2022': {45.0},
 'fastly inc_2018': {43.0},
 'freni brembo_2018': {131.0},
 'fuji soft inc_2022': {80.0, 81.0},
 'ge

In [259]:
retrieval_files = list((RETRIEVAL_DIR/METHOD).rglob("*.json"))

results = []

for retrieval_file in retrieval_files:
    # if retrieval_file.stem.split("_")[0] != "amano corp":
    #     continue
    with open(retrieval_file, "r", encoding="utf-8") as f:
        rdata = json.load(f)
    
    
    retrieval_method = rdata.get("method")
    
    report_id = rdata.get("report_id")
    if report_id not in gold_pages_by_report:
        continue
    
    retrieved_results = rdata.get("items", [])
    retrieved_pages = set()
    
    for item in retrieved_results:
        metadata = item.get("metadata", {})
        page_numbers = metadata.get("page_no")
        retrieved_pages.add(page_numbers)
    
    gold_pages = gold_pages_by_report[report_id]
    
    # Evaluate retrieval results
    true_positives = len(retrieved_pages & gold_pages)
    false_positives = len(retrieved_pages - gold_pages)
    false_negatives = len(gold_pages - retrieved_pages)
    
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    results.append({
        "report_id": report_id,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score
    })
 
print(f"Retrieval Method: {retrieval_method}")
df = pd.DataFrame(results)


if NO_GRAPHICS:
    df.to_csv(f"{RESULTS_DIR}/{METHOD}_retrieval_eval_no_graphics.csv", index=False)
else:
    df.to_csv(f"{RESULTS_DIR}/{METHOD}_retrieval_eval.csv", index=False)
df

Retrieval Method: hybrid


,report_id,precision,recall,f1_score
0,acuity brands inc_2022,0.4,1.0,0.571429
1,addtech_2022,0.2,1.0,0.333333
2,aixtron_2020,0.2,1.0,0.333333
3,allfunds group_2021,0.4,1.0,0.571429
4,Allianz_2022,0.2,1.0,0.333333
...,...,...,...,...
59,the simply good foods company_2018,0.2,1.0,0.333333
60,toshiba tec corp_2022,0.2,1.0,0.333333
61,uniper_2019,0.8,1.0,0.888889
62,varta ag_2021,0.2,1.0,0.333333


In [260]:
df["precision"].mean(), df["recall"].mean(), df["f1_score"].mean()

(np.float64(0.246875), np.float64(1.0), np.float64(0.3857266865079366))